# BoneCoT Inference Template

This notebook is a public release template for local BoneCoT inference.

Private annotation files and task-specific fine-tuned checkpoints are not distributed in this repository. Replace the placeholder paths below with your own local assets before running the notebook.


In [ ]:
import os
import pathlib

def resolve_repo_root():
    cwd = pathlib.Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / 'finetune').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root containing finetune/.')

REPO_ROOT = resolve_repo_root()
os.chdir(REPO_ROOT)
print(f'Repository root: {REPO_ROOT}')

CUDA_VISIBLE_DEVICES = '0'
BACKBONE_CHECKPOINT = REPO_ROOT / 'finetune' / 'checkpoints' / 'BoneFM.pth'
TEST_DATASET_CSV = '/path/to/private/test_dataset.csv'
MAIN_TASK = 'primary_metastatic'
TASK_CHECKPOINTS = {
    'primary_metastatic': '/path/to/private/primary_metastatic.pth',
    'osteoblastic': '/path/to/private/osteoblastic.pth',
    'osteolytic': '/path/to/private/osteolytic.pth',
    'spinal_cord_compression': '/path/to/private/spinal_cord_compression.pth',
    'pathological_fracture': '/path/to/private/pathological_fracture.pth',
    'type_of_primary_tumor': '/path/to/private/type_of_primary_tumor.pth',
}
OUTPUT_DIR = REPO_ROOT / 'finetune_logs' / 'bonecot_inference'


## Validate Private Assets

Run this cell after replacing the placeholder paths.

In [ ]:
from pathlib import Path

def validate_private_assets():
    missing = []
    if not BACKBONE_CHECKPOINT.exists():
        missing.append(f'backbone checkpoint: {BACKBONE_CHECKPOINT}')
    dataset_csv = Path(TEST_DATASET_CSV)
    if not dataset_csv.exists():
        missing.append(f'test dataset csv: {dataset_csv}')
    for task_name, ckpt_path in TASK_CHECKPOINTS.items():
        if not Path(ckpt_path).exists():
            missing.append(f'{task_name} checkpoint: {ckpt_path}')
    if missing:
        raise FileNotFoundError('Missing required private assets\n' + '\n'.join(missing))
    print('All required private assets are available.')

validate_private_assets()


## Run BoneCoT Inference

In [ ]:
import argparse
from omegaconf import OmegaConf

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES

from finetune.trainer import BoneCoT_Eval_Trainer

parser = argparse.ArgumentParser(description='BoneCoT inference')
parser.add_argument('--config-file', type=str, default=str(REPO_ROOT / 'finetune' / 'configs' / 'bonecot_eval.yaml'))
parser.add_argument('--output-dir', type=str, default=str(OUTPUT_DIR))
parser.add_argument('--log-interval', type=int, default=10)
parser.add_argument('--log_display', action='store_true', default=True)
args = parser.parse_args([])

default_cfg = OmegaConf.load(REPO_ROOT / 'finetune' / 'configs' / 'default_configs.yaml')
user_cfg = OmegaConf.load(args.config_file)
merged_cfg = OmegaConf.merge(default_cfg, user_cfg)
for key, value in merged_cfg.items():
    key_str = str(key)
    if not hasattr(args, key_str) or getattr(args, key_str) is None:
        setattr(args, key_str, value)

args.output_dir = os.path.abspath(args.output_dir)
os.makedirs(args.output_dir, exist_ok=True)
args.data.train_dataset = TEST_DATASET_CSV
args.data.val_dataset = TEST_DATASET_CSV
args.data.test_dataset = TEST_DATASET_CSV
args.model.backbone_ckpt_path = str(BACKBONE_CHECKPOINT)
args.model.main_task = MAIN_TASK
for task_name, ckpt_path in TASK_CHECKPOINTS.items():
    args.model.model_ckpt_dict[task_name] = ckpt_path
args.model.relative_task = [task for task in args.model.relative_task if task != args.model.main_task]

bonecot_eval_trainer = BoneCoT_Eval_Trainer(args)
bonecot_eval_trainer.run()


## Inspect Saved Prediction Files

In [ ]:
pred_dir = OUTPUT_DIR / 'pred_npz'
if pred_dir.exists():
    print('Generated prediction files:')
    for path in sorted(pred_dir.glob('*.npz')):
        print(path)
else:
    print(f'Prediction directory not found: {pred_dir}')
